# Combined coolin operation optimization

In [ ]:
# - [ ] Model water availability in reservoir as a function of precipitations


In [ ]:
from itertools import product
# Try with validation data
df = pd.read_csv(Path("../../modeling/assets/data.csv"), )
# df = df[(df["Rp"]>0.9)] #& (df["Rs"]<0.1)]

# ds = df.iloc[0]
display(df)
detailed_list = []

for idx, (price_factor, df_idx) in enumerate(product([1000, 100, 1], range(len(df)))):
    ds = df.iloc[df_idx]
    env_vars = EnvironmentVariables(
        HR=ds["HR"],
        Tamb=ds["Tamb"],
        Q=ds["Qreleased"],
        # mv=ds["Pth_kW"] / (w_props(T=ds["Tv_C"]+273.15, x=1).h - w_props(T=ds["Tv_C"]+273.15, x=0).h) * 3600,
        Tv=ds["Tv"],
        Pw_s1=df_env["water_price_morocco_eur_l"]*price_factor,
        Pw_s2=df_env["water_price_morocco_alternative_eur_l"]*price_factor,
        Pe=cost_e,
        Vavail=df_env["Vavail_l"]
    )

    problem = Problem(env_vars=env_vars, debug_mode=False)

    if idx == 0:
        # Evalute prior to optimization (using experimental values), only once
        dv = problem.decision_vector_to_decision_variables([ds["qc"], ds["wwct"]])
        # display(dv)
        ev = env_vars.to_matlab()
        _, _, detailed = cc_model.combined_cooler_model(ev.Tamb, ev.HR, ev.mv, dv.qc, dv.Rp, dv.Rs, dv.wdc, dv.wwct, ev.Tv, nargout=3)
        detailed_list.append(detailed)

    # Evaluate optimization
    detailed_list.append(optimize(problem, extra_outputs=False, log_verbosity=0, n_trials=1)[0])

df_results = pd.DataFrame(detailed_list)
display(df_results)

fig = plot_hydraulic_distribution(df_results["qc"].values, df_results["Rp"].values, df_results["Rs"].values)
display(fig)
